# Probabilistic Analysis and Randomized Algorithms

This notebook is about what changes when randomness enters algorithm analysis. Sometimes the algorithm is deterministic and the input order is random. Sometimes the algorithm itself makes random choices.

The core case study is the hiring-assistant problem from CLRS: a deterministic online rule whose expected cost can be analyzed using indicator random variables. After the core lesson, optional bonus sections connect the same ideas to secretary problems, randomized quicksort, Monte Carlo algorithms, and graph cuts.

Main text source: Cormen, Leiserson, Rivest, and Stein, *Introduction to Algorithms*, 4th edition, Chapter 5, [MIT Press](https://mitpress.mit.edu/9780262367509/introduction-to-algorithms/).


## Learning Goals

By the end of the core part of this notebook you should be able to:

- connect randomized algorithms to earlier topics from the course;
- distinguish probabilistic analysis from randomized algorithm design;
- use indicator random variables and linearity of expectation in a simple algorithm analysis;
- derive why the hiring-assistant algorithm hires about $H_n = \Theta(\log n)$ times on a random input order;
- run Monte Carlo experiments that estimate expected cost and compare hiring rules;
- classify Las Vegas and Monte Carlo randomized algorithms in the bonus examples.


## Bridge From Earlier Topics

This topic comes late in the course, so use it as a bridge rather than as an isolated chapter.

- **Basic analysis**: we still count operations and costs, but now we often ask for expected cost instead of only worst-case cost.
- **Searching and sorting**: randomized quicksort uses random pivots to avoid a fixed bad input pattern.
- **Greedy algorithms**: an online hiring rule is a greedy rule; it makes the best local decision from the information currently available.
- **Dynamic programming and backtracking**: those methods usually search a structured state space deterministically; randomized algorithms often trade complete control for good expected behavior.
- **Graph algorithms**: Karger's min-cut algorithm uses random contraction and succeeds with some probability, then repetition amplifies that probability.
- **String matching**: randomized fingerprints and hashing are common examples where a fast answer may have a small false-positive probability.

The shared question is: what guarantee do we get when some part of the computation or input model is random?


## Lesson Route: Core First, Bonus If Time Allows

For a `2 x 45` minute class, the priority path is:

1. Define the hiring-assistant model and its costs.
2. Analyze expected hires with indicator random variables.
3. Implement the hiring rule.
4. Run experiments for random order, sorted order, and threshold rules.
5. Use Monte Carlo simulation to estimate expected cost.

The later sections are marked **Bonus**. They are useful extensions, but the notebook still works as a complete lesson if you stop after the hiring experiments and the terminology summary.


## Hiring Assistant Problem

Suppose you need to hire a new office assistant. An employment agency sends one candidate each day. You interview candidates in arrival order, and after each interview you must either keep the current assistant or hire the new candidate.

### Model

- There are `n` candidates.
- Candidate quality can be ranked from worst to best. In the code we model quality as a numeric score.
- Every interview costs `interview_cost`.
- Every hire costs `hire_cost`.
- Replacing an existing assistant also costs `fire_cost`.
- The basic rule is: hire a candidate if they are better than every candidate seen so far.

The algorithm is deterministic once the candidate order is fixed. The probabilistic part is the assumption that the candidates arrive in a uniformly random order.

### Question

How many times should we expect to hire someone? This matters because interviews happen `n` times, but hiring and replacing are the expensive events.


Source notes for this section:

- CLRS Chapter 5.1 presents the hiring-assistant problem as the standard introduction to probabilistic analysis.
- Broader hiring variants are summarized in [sources/randomized_algorithms/hiring_problem.md](../sources/randomized_algorithms/hiring_problem.md), based on Sergei Vassilvitskii's Stanford slide deck, [The Hiring Problem: Going Beyond Secretaries](https://theory.stanford.edu/~sergei/slides/hiring-dagstuhl.pdf).


### Hiring Rule Pseudocode

The basic online replacement rule is:

1. Start with no assistant and no best score.
2. Interview candidates one at a time.
3. If the new candidate is better than the current best candidate:
   - hire the new candidate;
   - if there was already an assistant, pay the replacement/firing cost;
   - update the current best candidate and score.
4. Otherwise, reject the candidate.
5. Return the total cost, number of hires, and best score.

The exact scoring system is part of the model. In the code below, a candidate is represented directly by a numeric score.


### Why the Cost Model Matters

If interviews and replacements are free, we can scan all candidates and return the maximum score in `O(n)` time. The interesting question appears when each interview and each replacement has a cost, because then the number of times we hire matters.


In [ ]:
from __future__ import annotations

from collections import Counter
import math
import random
import statistics

try:
    import numpy as np
except Exception:
    np = None

try:
    import pandas as pd
except Exception:
    pd = None

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

try:
    from IPython import get_ipython
    from IPython.display import display
    import ipywidgets as widgets
    NOTEBOOK_UI = get_ipython() is not None
except Exception:
    display = print
    widgets = None
    NOTEBOOK_UI = False

WIDGETS_AVAILABLE = widgets is not None and NOTEBOOK_UI
SEED = 2025
random.seed(SEED)
rng = random.Random(SEED)


def harmonic_number(n: int) -> float:
    """Return H_n = 1 + 1/2 + ... + 1/n."""
    if n < 1:
        return 0.0
    return sum(1 / k for k in range(1, n + 1))


def show_table(rows):
    """Display rows nicely when pandas is available, otherwise return plain data."""
    if pd is not None:
        return pd.DataFrame(rows)
    return rows


print(f"Using random seed {SEED}")


## Probabilistic Analysis Toolkit

For the first analysis, we only need two ideas.

An **indicator random variable** is `1` when an event happens and `0` otherwise. Let `I_i` be `1` when candidate `i` is hired, and `0` otherwise. Then the total number of hires is:

`X = I_1 + I_2 + ... + I_n`.

By **linearity of expectation**:

`E[X] = E[I_1] + E[I_2] + ... + E[I_n]`.

Candidate `i` is hired exactly when they are better than all candidates before them. In a random permutation, among the first `i` candidates, each candidate is equally likely to be the best. Therefore:

`P(candidate i is hired) = 1/i`.

So the expected number of hires is:

$E[X] = 1 + 1/2 + 1/3 + \cdots + 1/n = H_n = \Theta(\log n)$.

The hires are the **records** we see while scanning the candidate sequence from left to right.


In [ ]:
def count_records(order):
    """Count how many times a new maximum appears while scanning left to right."""
    best_so_far = -math.inf
    records = 0
    for value in order:
        if value > best_so_far:
            records += 1
            best_so_far = value
    return records


def simulate_record_counts(n: int, trials: int = 10_000, seed: int = SEED):
    local_rng = random.Random(seed)
    base_order = list(range(n))
    counts = []
    for _ in range(trials):
        order = base_order[:]
        local_rng.shuffle(order)
        counts.append(count_records(order))
    return {
        "n": n,
        "trials": trials,
        "empirical_mean_hires": sum(counts) / trials,
        "theoretical_H_n": harmonic_number(n),
        "min_hires_seen": min(counts),
        "max_hires_seen": max(counts),
    }


record_rows = [simulate_record_counts(n, trials=5_000) for n in (10, 25, 50, 100, 250)]
show_table(record_rows)


In [ ]:
if plt is not None:
    xs = [row["n"] for row in record_rows]
    empirical = [row["empirical_mean_hires"] for row in record_rows]
    theory = [row["theoretical_H_n"] for row in record_rows]

    plt.figure(figsize=(7, 4))
    plt.plot(xs, empirical, "o-", label="simulation")
    plt.plot(xs, theory, "s--", label="H_n theory")
    plt.xlabel("number of candidates")
    plt.ylabel("expected number of hires")
    plt.title("Hiring assistant: records in a random order")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()
else:
    print("matplotlib is not available, skipping plot")


### Hiring Assistant Implementation Notes

The implementation scans candidates once. Every candidate costs one interview. A candidate is hired when the selected rule returns `True`. The first hire pays only the hiring cost; later hires also pay the firing cost for replacing the current assistant.

The function returns a dictionary instead of a tuple so that the result is easy to read in notebooks: total cost, best candidate, best score, number of hires, and number of replacements.

With the default rule, every new record is hired. On a random candidate order, the expected number of such records is $H_n$, so the expensive part of the process grows like $\Theta(\log n)$ in expectation, not $\Theta(n)$.


In [ ]:
def score(candidate):
    """In this teaching model, the candidate object is already its score."""
    return candidate


def is_hirable(candidate_score, index, num_candidates, best_score, hire_cost, fire_cost, total_cost, max_budget):
    """Hire whenever the current candidate is the best seen so far."""
    return candidate_score > best_score


def hire_assistant(
    candidates,
    is_hirable_fun=is_hirable,
    hire_cost=0,
    fire_cost=0,
    interview_cost=0,
    max_budget=0,
    max_hires=None,
    debug=False,
):
    """Run an online hiring rule and return a readable result dictionary."""
    best_candidate = None
    best_score = -math.inf
    total_cost = 0
    hire_count = 0
    replacement_count = 0
    n = len(candidates)

    for index, candidate in enumerate(candidates):
        candidate_score = score(candidate)
        total_cost += interview_cost

        if is_hirable_fun(candidate_score, index, n, best_score, hire_cost, fire_cost, total_cost, max_budget):
            if best_candidate is not None:
                total_cost += fire_cost
                replacement_count += 1
                if debug:
                    print(f"Paying {fire_cost} to replace candidate with score {best_score}")

            total_cost += hire_cost
            best_candidate = candidate
            best_score = candidate_score
            hire_count += 1

            if debug:
                print(f"Hiring candidate with score {candidate_score}, paying {hire_cost}")
                print(f"Total cost now is {total_cost}")

            if max_hires is not None and hire_count >= max_hires:
                break

    return {
        "total_cost": total_cost,
        "best_candidate": best_candidate,
        "best_score": None if best_candidate is None else best_score,
        "hire_count": hire_count,
        "replacement_count": replacement_count,
        "interviews": index + 1 if n else 0,
    }


In [ ]:
random.seed(SEED)
candidates = [round(random.random(), 4) for _ in range(20)]
candidates


In [ ]:
# Zero-cost scan: we interview everyone and eventually keep the best candidate seen.
hire_assistant(candidates)


## Known Score Range and Threshold Rules

If the score distribution is known, we might set a threshold such as "hire the first candidate above `0.90`". This can reduce replacement costs, but it changes the risk: a threshold that is too high may hire nobody, while a threshold that is too low may still replace often.

In real hiring settings the score range and distribution are usually not known this cleanly. Here the simplified model lets us study the trade-off.


In [ ]:
# how many candidates do we have?
print(f"We have {len(candidates)} candidates")


In [ ]:
hire_assistant(candidates,
               is_hirable_fun=is_hirable,
               hire_cost=2_000,
               fire_cost=5_000,
               interview_cost=250)


## Worst Case Arrival Order: Sorted Candidates

The random-order assumption matters. If candidates arrive from worst to best, the basic rule hires every candidate, which is the expensive worst case for replacement costs.


In [ ]:
# employment agency gives you a list of candidates in already sorted order,
# sadly for you it is ascending and you do not realize that
sorted_candidates = sorted(candidates)
hire_assistant(sorted_candidates, is_hirable_fun=is_hirable,
               hire_cost=2_000,
               fire_cost=5_000,
               interview_cost=200)


### Threshold Rules

A fixed threshold can be a reasonable decision rule when the score distribution is known: hire a candidate only if their score is high enough. The rule itself is deterministic, but its cost and success probability still need probabilistic analysis because candidate order and candidate scores may be random.


In [ ]:
def is_hirable_thresh(candidate_score, index, num_candidates, best_score, hire_cost, fire_cost, total_cost, max_budget, threshold=0.9):
    return candidate_score > best_score and candidate_score > threshold


hire_assistant(
    sorted_candidates,
    is_hirable_fun=is_hirable_thresh,
    hire_cost=2_000,
    fire_cost=5_000,
    interview_cost=200,
)


In [ ]:
# If we are allowed to hire only one person, then a threshold rule can stop early.
hire_assistant(
    sorted_candidates,
    is_hirable_fun=is_hirable_thresh,
    hire_cost=2_000,
    fire_cost=5_000,
    interview_cost=200,
    max_hires=1,
)


In [ ]:
def make_hirable_function_with_threshold(threshold_value):
    """Create an is_hirable function with a fixed threshold."""
    def is_hirable_with_custom_thresh(candidate_score, index, num_candidates, best_score, hire_cost, fire_cost, total_cost, max_budget):
        return candidate_score > best_score and candidate_score > threshold_value
    return is_hirable_with_custom_thresh


thresh_07_hirable_func = make_hirable_function_with_threshold(0.7)
hire_assistant(candidates, is_hirable_fun=thresh_07_hirable_func, hire_cost=2_000, fire_cost=5_000, interview_cost=200)


## Why Monte Carlo Estimates Stabilize

Monte Carlo simulation estimates quantities by repeating random experiments and averaging the results. The **law of large numbers** says that, as the number of independent trials grows, the sample average tends to move toward the true expected value.

The dice example below is only a warm-up. The same averaging idea is used afterward to estimate the expected cost of hiring.

![Fair Dice](https://github.com/ValRCS/RTU_Algorithms_DIP321/blob/main/imgs/Lawoflargenumbers.svg.png?raw=true)


In [ ]:
random.seed(SEED)
throw_dict = {}
for throws in [1, 10] + list(range(100, 100_100, 100)):
    dice_throws = [random.randint(1, 6) for _ in range(throws)]
    avg = sum(dice_throws) / throws
    throw_dict[str(throws)] = avg
    if throws in [10, 100, 1_000, 10_000, 100_000]:
        print(f"Average die roll after {throws} throws is {avg}")


In [ ]:
if plt is not None:
    x = [int(key) for key in throw_dict.keys()]
    plt.plot(x, throw_dict.values())
    plt.axhline(3.5, color="tab:red", linestyle="--", label="expected value")
    plt.xlabel("throws")
    plt.ylabel("average die roll")
    plt.title("Law of large numbers for a fair die")
    plt.xticks([1_000, 10_000, 100_000], ["1k", "10k", "100k"])
    plt.legend()
    plt.show()
else:
    print("matplotlib is not available, skipping plot")


## Monte Carlo Simulation for Hiring Cost

For the hiring-assistant problem, Monte Carlo simulation means:

1. Generate many random candidate orders.
2. Run the same hiring rule on each order.
3. Record cost, number of hires, and final selected score.
4. Average those results.

This does not replace the analysis. It gives an empirical check and lets us compare variants, such as different thresholds, when the algebra becomes less convenient.


### Sampling Candidate Orders

`random.sample(candidates, len(candidates))` returns a random order without replacement, which is exactly what we need for one simulated arrival order.


In [ ]:
random.sample([1, 2, 3, 4], 4)  # random order without replacement


In [ ]:
def hire_assistant_simulate(
    candidates,
    hire_cost,
    fire_cost,
    interview_cost,
    num_simulations,
    is_hirable_fun=is_hirable,
    max_budget=0,
    debug=False,
    seed=SEED,
):
    total_cost = 0
    total_hires = 0
    total_replacements = 0
    selected_scores = []
    n = len(candidates)
    local_rng = random.Random(seed)

    for _ in range(num_simulations):
        candidate_pool = local_rng.sample(candidates, n)
        result = hire_assistant(
            candidate_pool,
            is_hirable_fun,
            hire_cost,
            fire_cost,
            interview_cost,
            max_budget=max_budget,
            debug=debug,
        )
        total_cost += result["total_cost"]
        total_hires += result["hire_count"]
        total_replacements += result["replacement_count"]
        if result["best_score"] is not None:
            selected_scores.append(result["best_score"])

    return {
        "simulations": num_simulations,
        "average_cost": total_cost / num_simulations,
        "average_hires": total_hires / num_simulations,
        "average_replacements": total_replacements / num_simulations,
        "hire_rate": len(selected_scores) / num_simulations,
        "average_selected_score": statistics.mean(selected_scores) if selected_scores else None,
    }


In [ ]:
# Basic rule: every new record is hired. We always end with the best candidate in the pool.
baseline_simulation = hire_assistant_simulate(
    candidates,
    hire_cost=2_000,
    fire_cost=5_000,
    interview_cost=200,
    num_simulations=10_000,
)
baseline_simulation


In [ ]:
# Threshold 0.90: fewer hires, but possibly no hire if no candidate clears the threshold.
threshold_09_simulation = hire_assistant_simulate(
    candidates,
    hire_cost=2_000,
    fire_cost=5_000,
    interview_cost=200,
    num_simulations=10_000,
    is_hirable_fun=is_hirable_thresh,
)
threshold_09_simulation


In [ ]:
def is_hirable_thresh_07(candidate_score, index, num_candidates, best_score, hire_cost, fire_cost, total_cost, max_budget, threshold=0.7):
    return candidate_score > best_score and candidate_score > threshold


threshold_07_simulation = hire_assistant_simulate(
    candidates,
    hire_cost=2_000,
    fire_cost=5_000,
    interview_cost=200,
    num_simulations=10_000,
    is_hirable_fun=is_hirable_thresh_07,
)
threshold_07_simulation


In [ ]:
threshold_rows = []
for threshold in (0.50, 0.70, 0.80, 0.90, 0.95):
    threshold_rule = make_hirable_function_with_threshold(threshold)
    result = hire_assistant_simulate(
        candidates,
        hire_cost=2_000,
        fire_cost=5_000,
        interview_cost=200,
        num_simulations=10_000,
        is_hirable_fun=threshold_rule,
    )
    threshold_rows.append({
        "threshold": threshold,
        "average_cost": result["average_cost"],
        "average_hires": result["average_hires"],
        "hire_rate": result["hire_rate"],
        "average_selected_score": result["average_selected_score"],
    })

show_table(threshold_rows)


## Main Terms: Probabilistic Analysis, Las Vegas, Monte Carlo

### Probabilistic Analysis

The algorithm may be deterministic, but the input model is random. The hiring-assistant analysis is the main example: once the candidate order is fixed, the rule has no random choices.

### Las Vegas Algorithms

A Las Vegas algorithm always returns a correct answer, but its running time may depend on random choices. Randomized quicksort is the standard example: the sorted output is correct, while random pivots make the expected running time good on every fixed input.

### Monte Carlo Algorithms

A Monte Carlo algorithm has a bounded running time, but it may return an incorrect answer with small probability. The error can be one-sided, where only one kind of answer can be wrong, or two-sided, where either answer might be wrong.

Examples include Freivalds' algorithm for checking matrix multiplication and probabilistic primality tests such as Miller-Rabin. Independent repetition is a common way to reduce the error probability.


## Bonus: Secretary Problem and Optimal Stopping

The secretary problem is an example of an **optimal stopping problem**: after each observation, the algorithm must either stop and accept the current candidate or continue and lose that candidate forever.

The classical secretary problem is stricter than the hiring-assistant problem:

- candidates arrive one at a time in random order;
- after each interview we know only the candidate's rank relative to the candidates already seen;
- we may hire exactly one candidate;
- rejected candidates cannot be recalled;
- the goal is to maximize the probability of hiring the single best candidate.

The standard **look-then-leap** strategy is:

1. Observe the first `r` candidates, but hire none of them.
2. Remember the best candidate from this observation phase.
3. Hire the first later candidate who is better than everyone observed so far.
4. If no such candidate appears, hire the last candidate.

For large `n`, choosing `r` close to `n/e` maximizes the success probability. If `t = r/n`, the approximate success probability is `-t ln(t)`, which is maximized at `t = 1/e` and has value about `1/e = 0.3679`.


In [ ]:
def secretary_select(order, skip):
    """Return the selected index and value using the look-then-leap rule."""
    n = len(order)
    if n == 0:
        raise ValueError("order must contain at least one candidate")

    skip = max(0, min(skip, n - 1))
    best_observed = max(order[:skip], default=-math.inf)

    for index in range(skip, n):
        if order[index] > best_observed:
            return index, order[index]

    return n - 1, order[-1]


def secretary_theoretical_success(n, skip):
    """Exact success probability for the look-then-leap rule with n candidates."""
    if n <= 0:
        raise ValueError("n must be positive")

    skip = max(0, min(skip, n - 1))
    if skip == 0:
        return 1 / n

    return (skip / n) * sum(1 / (position - 1) for position in range(skip + 1, n + 1))


def secretary_asymptotic_success(skip_fraction):
    """Large-n approximation -t ln(t), where t is the skipped fraction."""
    if skip_fraction <= 0 or skip_fraction >= 1:
        return 0.0
    return -skip_fraction * math.log(skip_fraction)


def simulate_secretary_strategy(n, skip, trials=10_000, seed=SEED):
    local_rng = random.Random(seed)
    base_order = list(range(n))
    successes = 0
    selected_positions = []
    selected_ranks = []

    for _ in range(trials):
        order = base_order[:]
        local_rng.shuffle(order)
        selected_index, selected_value = secretary_select(order, skip)
        successes += selected_value == n - 1
        selected_positions.append(selected_index + 1)
        selected_ranks.append(n - selected_value)

    return {
        "n": n,
        "skip": skip,
        "skip_fraction": skip / n,
        "trials": trials,
        "empirical_success": successes / trials,
        "theoretical_success": secretary_theoretical_success(n, skip),
        "asymptotic_success": secretary_asymptotic_success(skip / n),
        "average_selected_position": sum(selected_positions) / trials,
        "average_selected_rank": sum(selected_ranks) / trials,
    }


n = 100
skip = round(n / math.e)
simulate_secretary_strategy(n, skip, trials=20_000)


In [ ]:
def secretary_success_curve(n=100, trials=3_000, step=2, seed=SEED):
    rows = []
    for skip in range(0, n, step):
        rows.append(simulate_secretary_strategy(n, skip, trials=trials, seed=seed + skip))
    return rows


curve_rows = secretary_success_curve(n=100, trials=3_000, step=2)
best_empirical = max(curve_rows, key=lambda row: row["empirical_success"])
best_theoretical = max(curve_rows, key=lambda row: row["theoretical_success"])

print(f"Best empirical skip in sampled grid: {best_empirical['skip']} / 100")
print(f"Best theoretical skip in sampled grid: {best_theoretical['skip']} / 100")
print(f"n/e suggests skipping about {round(100 / math.e)} candidates")

show_table([
    row for row in curve_rows
    if row["skip"] in {0, 10, 20, 30, 36, 38, 50, 70, 90}
])


In [ ]:
if plt is not None:
    skip_values = [row["skip"] for row in curve_rows]
    empirical = [row["empirical_success"] for row in curve_rows]
    theoretical = [row["theoretical_success"] for row in curve_rows]

    plt.figure(figsize=(8, 4.5))
    plt.plot(skip_values, empirical, "o", label="simulation")
    plt.plot(skip_values, theoretical, "-", label="exact theory")
    plt.axvline(100 / math.e, color="tab:red", linestyle="--", label="n/e")
    plt.xlabel("candidates skipped before hiring is allowed")
    plt.ylabel("probability of selecting the best candidate")
    plt.title("Secretary problem: look-then-leap strategy")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()
else:
    print("matplotlib is not available, skipping plot")


In [ ]:
def show_secretary_experiment(n=100, skip_fraction=1 / math.e, trials=2_000):
    skip = round(n * skip_fraction)
    result = simulate_secretary_strategy(n=n, skip=skip, trials=trials)
    print(f"n = {n}, skip = {skip}, trials = {trials}")
    print(f"empirical success:   {result['empirical_success']:.4f}")
    print(f"exact theory:        {result['theoretical_success']:.4f}")
    print(f"large-n estimate:    {result['asymptotic_success']:.4f}")
    print(f"average rank chosen: {result['average_selected_rank']:.2f} (1 means best)")


if WIDGETS_AVAILABLE:
    widgets.interact(
        show_secretary_experiment,
        n=widgets.IntSlider(value=100, min=10, max=300, step=10),
        skip_fraction=widgets.FloatSlider(value=1 / math.e, min=0.0, max=0.95, step=0.01, readout_format=".2f"),
        trials=widgets.IntSlider(value=2_000, min=500, max=10_000, step=500),
    )
else:
    show_secretary_experiment()


### Bonus: Other Versions of the Secretary Problem

The classical version is only one model. Common variants include:

- **Multiple hires**: choose up to `k` candidates instead of exactly one.
- **Hiring and firing**: allow replacement, but charge a cost for each replacement.
- **Unknown number of applicants**: make decisions without knowing `n` in advance.
- **Adversarial order**: remove the random-arrival assumption and study what guarantees remain.
- **Distributed selection**: coordinate several decision-makers or offices.

These variants connect the secretary problem to online algorithms, optimal stopping, and realistic hiring models such as the Stanford hiring-problem slides summarized in `sources/randomized_algorithms/hiring_problem.md`.


## Bonus: Beyond One Hire - Adaptive Hiring Strategies

The classical secretary problem asks for exactly one hire. The source note in `sources/randomized_algorithms/hiring_problem.md` points to a broader model: an organization may hire many people over time while balancing interview volume and average quality.

We will compare four simple strategies on candidates whose quality is sampled uniformly from `[0, 1)`:

| Strategy | Rule | Main trade-off |
|---|---|---|
| Fixed threshold | hire if `quality >= threshold` | simple, but quality stops improving |
| New record | hire only if better than everyone hired so far | high quality, but increasingly slow |
| Above mean | hire if better than current hired average | adaptive and moderately fast |
| Above median | hire if better than current hired median | stricter than mean once the team improves |

These are teaching simulations, not production hiring recommendations. Their purpose is to make the probability and cost trade-offs visible.


In [ ]:
def should_hire_adaptive(strategy, quality, hired, threshold=0.8):
    if strategy == "threshold":
        return quality >= threshold
    if not hired:
        return True
    if strategy == "record":
        return quality > max(hired)
    if strategy == "above_mean":
        return quality > statistics.mean(hired)
    if strategy == "above_median":
        return quality > statistics.median(hired)
    raise ValueError(f"unknown strategy: {strategy}")


def simulate_adaptive_hiring(
    strategy,
    target_hires=40,
    threshold=0.8,
    max_interviews=20_000,
    seed=SEED,
):
    local_rng = random.Random(seed)
    hired = []
    quality_history = []
    interview_history = []

    for interviews in range(1, max_interviews + 1):
        quality = local_rng.random()
        if should_hire_adaptive(strategy, quality, hired, threshold):
            hired.append(quality)
            quality_history.append(statistics.mean(hired))
            interview_history.append(interviews)
            if len(hired) >= target_hires:
                break

    return {
        "strategy": strategy,
        "target_hires": target_hires,
        "hires": len(hired),
        "interviews": interview_history[-1] if interview_history else max_interviews,
        "reached_target": len(hired) >= target_hires,
        "average_quality": statistics.mean(hired) if hired else 0.0,
        "best_quality": max(hired) if hired else 0.0,
        "quality_history": quality_history,
        "interview_history": interview_history,
    }


def compare_adaptive_hiring(target_hires=40, threshold=0.8, trials=50):
    strategies = ["threshold", "record", "above_mean", "above_median"]
    rows = []
    for strategy in strategies:
        runs = [
            simulate_adaptive_hiring(
                strategy,
                target_hires=target_hires,
                threshold=threshold,
                seed=SEED + 10_000 * trial + len(strategy),
            )
            for trial in range(trials)
        ]
        reached = [run for run in runs if run["reached_target"]]
        rows.append({
            "strategy": strategy,
            "target_hires": target_hires,
            "target_reached_rate": len(reached) / trials,
            "avg_interviews_used": statistics.mean(run["interviews"] for run in runs),
            "avg_final_quality": statistics.mean(run["average_quality"] for run in runs),
            "avg_best_quality": statistics.mean(run["best_quality"] for run in runs),
        })
    return rows


adaptive_rows = compare_adaptive_hiring(target_hires=40, threshold=0.8, trials=50)
show_table(adaptive_rows)


In [ ]:
if plt is not None:
    example_runs = [
        simulate_adaptive_hiring(strategy, target_hires=40, threshold=0.8, seed=SEED + index)
        for index, strategy in enumerate(["threshold", "record", "above_mean", "above_median"])
    ]

    plt.figure(figsize=(8, 4.8))
    for run in example_runs:
        plt.plot(run["interview_history"], run["quality_history"], marker="o", markersize=3, label=run["strategy"])
    plt.xlabel("interviews used")
    plt.ylabel("average quality among hires")
    plt.title("Adaptive hiring strategies: one simulated run")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()
else:
    print("matplotlib is not available, skipping plot")


In [ ]:
def show_adaptive_hiring_experiment(target_hires=40, threshold=0.8, trials=50):
    rows = compare_adaptive_hiring(target_hires=target_hires, threshold=threshold, trials=trials)
    result = show_table(rows)
    display(result)


if WIDGETS_AVAILABLE:
    widgets.interact(
        show_adaptive_hiring_experiment,
        target_hires=widgets.IntSlider(value=40, min=10, max=100, step=10),
        threshold=widgets.FloatSlider(value=0.8, min=0.5, max=0.98, step=0.02, readout_format=".2f"),
        trials=widgets.IntSlider(value=50, min=10, max=150, step=10),
    )
else:
    show_adaptive_hiring_experiment()


## Bonus: Monte Carlo Pi Estimation

To estimate pi with Monte Carlo sampling, generate random points in a square and count how many fall inside the inscribed circle. The ratio of hits inside the circle estimates the area ratio, so `pi` is approximately four times that ratio.

This is not a good practical way to compute pi, but it is useful for seeing convergence and sampling error.


In [ ]:
def estimate_pi(num_points, seed=SEED):
    local_rng = random.Random(seed)
    hits = 0
    for _ in range(num_points):
        x = local_rng.uniform(-1, 1)
        y = local_rng.uniform(-1, 1)
        if x * x + y * y <= 1:
            hits += 1
    estimate = 4 * hits / num_points
    p_hat = hits / num_points
    standard_error = 4 * math.sqrt(p_hat * (1 - p_hat) / num_points)
    return estimate, standard_error


pi_rows = []
for num_points in (100, 1_000, 10_000, 100_000):
    estimate, standard_error = estimate_pi(num_points)
    pi_rows.append({
        "points": num_points,
        "pi_estimate": estimate,
        "absolute_error": abs(math.pi - estimate),
        "approx_standard_error": standard_error,
    })

show_table(pi_rows)


In [ ]:
if plt is not None:
    xs = [row["points"] for row in pi_rows]
    estimates = [row["pi_estimate"] for row in pi_rows]
    errors = [row["approx_standard_error"] for row in pi_rows]

    plt.figure(figsize=(7, 4))
    plt.errorbar(xs, estimates, yerr=errors, fmt="o-", capsize=4, label="estimate +/- 1 SE")
    plt.axhline(math.pi, color="tab:red", linestyle="--", label="math.pi")
    plt.xscale("log")
    plt.xlabel("random points")
    plt.ylabel("pi estimate")
    plt.title("Monte Carlo pi convergence")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()
else:
    print("matplotlib is not available, skipping plot")


## Bonus: Monty Hall Problem

![Monty Hall doors](https://upload.wikimedia.org/wikipedia/commons/thumb/3/3f/Monty_open_door.svg/440px-Monty_open_door.svg.png)

The Monty Hall problem is a probability puzzle based on a game show. A contestant chooses one of three doors. One door hides a prize, and the other two doors hide goats. After the contestant chooses, the host opens one losing door and offers the contestant a chance to switch.

Switching wins with probability `2/3`; staying wins with probability `1/3`. The simulation below confirms the calculation.


### Correct Strategy for Monty Hall Problem

The initial choice has probability `1/3` of being correct and probability `2/3` of being wrong. When the host opens a losing door, switching wins exactly in the cases where the initial choice was wrong. Therefore switching wins with probability `2/3`.


In [ ]:
def monty_hall_simulation(switch):
    doors = ["goat", "goat", "car"]
    random.shuffle(doors)
    chosen_door = random.choice(doors)
    if chosen_door == "car":
        if switch: # so we chose the switch strategy and were unlucky to have chosen the car already - so we get goat
            return 0
        else: # no switch strategy - stay put
            return 1
    else: # when we have chosen a goat
        if switch: # we apply switch strategy
            return 1  # we win the car
        else:  # stay put strategy fails here - we end up with the goat
            return 0

num_simulations = 100_000
switch = True
wins = 0

for i in range(num_simulations):
    wins += monty_hall_simulation(switch)

print(f"Probability of winning with switch: {wins / num_simulations:.4f}")
print(f"Probability of winning without switch: {(num_simulations - wins) / num_simulations:.4f}")


In [ ]:
num_simulations = 100_000
switch = False
wins = 0

for i in range(num_simulations):
    wins += monty_hall_simulation(switch)

print(f"Probability of winning without switch: {wins / num_simulations:.4f}")
print(f"Probability of winning WITH switch: {(num_simulations - wins) / num_simulations:.4f}")


## Bonus: Jupyter `%%timeit`

Jupyter's `%%timeit` magic also uses repeated runs and summary statistics, although it is designed for timing code rather than estimating probabilities.


In [ ]:
%%timeit
sorted(list(range(100_000)))


## Bonus: Randomized Quicksort - A Las Vegas Example

Randomized quicksort is Las Vegas: the sorted result is always correct, but the number of comparisons depends on random pivot choices. Random pivots protect the algorithm from the fixed bad case where deterministic first-pivot quicksort receives already sorted input.


In [ ]:
def quicksort_with_count(values, pivot_rule="random", seed=SEED):
    local_rng = random.Random(seed)

    def sort_count(items):
        if len(items) <= 1:
            return list(items), 0

        if pivot_rule == "first":
            pivot_index = 0
        elif pivot_rule == "random":
            pivot_index = local_rng.randrange(len(items))
        else:
            raise ValueError("pivot_rule must be 'first' or 'random'")

        pivot = items[pivot_index]
        rest = items[:pivot_index] + items[pivot_index + 1:]
        less = [item for item in rest if item <= pivot]
        greater = [item for item in rest if item > pivot]

        sorted_less, left_comparisons = sort_count(less)
        sorted_greater, right_comparisons = sort_count(greater)
        return sorted_less + [pivot] + sorted_greater, len(rest) + left_comparisons + right_comparisons

    return sort_count(list(values))


def quicksort_comparison_rows(n=200, trials=200):
    sorted_input = list(range(n))
    shuffled_input = sorted_input[:]
    random.Random(SEED).shuffle(shuffled_input)

    first_sorted_comparisons = quicksort_with_count(sorted_input, pivot_rule="first")[1]
    first_shuffled_comparisons = quicksort_with_count(shuffled_input, pivot_rule="first")[1]
    random_sorted = [
        quicksort_with_count(sorted_input, pivot_rule="random", seed=SEED + trial)[1]
        for trial in range(trials)
    ]

    return [
        {"case": "first pivot, sorted input", "comparisons": first_sorted_comparisons},
        {"case": "first pivot, one shuffled input", "comparisons": first_shuffled_comparisons},
        {"case": "random pivot, sorted input, average", "comparisons": statistics.mean(random_sorted)},
        {"case": "random pivot, sorted input, min", "comparisons": min(random_sorted)},
        {"case": "random pivot, sorted input, max", "comparisons": max(random_sorted)},
    ]


quicksort_rows = quicksort_comparison_rows(n=200, trials=200)
show_table(quicksort_rows)


In [ ]:
if plt is not None:
    sizes = [25, 50, 100, 200, 400]
    first_pivot = []
    random_pivot = []
    for n in sizes:
        sorted_input = list(range(n))
        first_pivot.append(quicksort_with_count(sorted_input, pivot_rule="first")[1])
        random_counts = [
            quicksort_with_count(sorted_input, pivot_rule="random", seed=SEED + trial)[1]
            for trial in range(100)
        ]
        random_pivot.append(statistics.mean(random_counts))

    plt.figure(figsize=(7, 4.5))
    plt.plot(sizes, first_pivot, "o-", label="first pivot on sorted input")
    plt.plot(sizes, random_pivot, "s-", label="random pivot average")
    plt.xlabel("n")
    plt.ylabel("comparisons")
    plt.title("Random pivots avoid the fixed sorted-input bad case")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()
else:
    print("matplotlib is not available, skipping plot")


## Bonus: Freivalds' Algorithm - A Monte Carlo Example

Given square matrices `A`, `B`, and `C`, checking whether `A B = C` by recomputing `A B` costs cubic time with the basic matrix multiplication algorithm. Freivalds' algorithm checks a random vector `r` instead:

`A(B r) == C r`

If `A B = C`, the check always accepts. If `A B != C`, one random 0/1 vector misses the error with probability at most `1/2`. Repeating the test `k` independent times reduces the false-accept probability to at most `2^-k`.

This is a Monte Carlo algorithm with one-sided error.


In [ ]:
def mat_vec_mul(matrix, vector):
    return [sum(row[j] * vector[j] for j in range(len(vector))) for row in matrix]


def mat_mul(A, B):
    n = len(A)
    m = len(B[0])
    inner = len(B)
    return [
        [sum(A[i][k] * B[k][j] for k in range(inner)) for j in range(m)]
        for i in range(n)
    ]


def random_matrix(n, low=0, high=5, seed=SEED):
    local_rng = random.Random(seed)
    return [[local_rng.randint(low, high) for _ in range(n)] for _ in range(n)]


def freivalds_check(A, B, C, trials=1, seed=SEED):
    n = len(A)
    local_rng = random.Random(seed)
    for _ in range(trials):
        r = [local_rng.randint(0, 1) for _ in range(n)]
        Br = mat_vec_mul(B, r)
        ABr = mat_vec_mul(A, Br)
        Cr = mat_vec_mul(C, r)
        if ABr != Cr:
            return False
    return True


A = random_matrix(4, seed=SEED)
B = random_matrix(4, seed=SEED + 1)
C = mat_mul(A, B)
C_bad = [row[:] for row in C]
C_bad[0][0] += 1

print("Correct product accepted:", freivalds_check(A, B, C, trials=5))
print("Incorrect product accepted:", freivalds_check(A, B, C_bad, trials=5))


In [ ]:
def freivalds_false_accept_rate(n=5, trials_per_check=1, experiments=2_000, seed=SEED):
    local_rng = random.Random(seed)
    false_accepts = 0
    for experiment in range(experiments):
        A = random_matrix(n, seed=local_rng.randrange(10**9))
        B = random_matrix(n, seed=local_rng.randrange(10**9))
        C = mat_mul(A, B)
        row = local_rng.randrange(n)
        col = local_rng.randrange(n)
        C[row][col] += 1
        if freivalds_check(A, B, C, trials=trials_per_check, seed=local_rng.randrange(10**9)):
            false_accepts += 1
    return false_accepts / experiments


freivalds_rows = []
for trial_count in range(1, 8):
    freivalds_rows.append({
        "trials": trial_count,
        "empirical_false_accept_rate": freivalds_false_accept_rate(trials_per_check=trial_count),
        "upper_bound_2^-k": 2 ** (-trial_count),
    })

show_table(freivalds_rows)


In [ ]:
def show_freivalds_experiment(matrix_size=5, trials_per_check=3, experiments=1_000):
    rate = freivalds_false_accept_rate(
        n=matrix_size,
        trials_per_check=trials_per_check,
        experiments=experiments,
    )
    print(f"matrix size: {matrix_size} x {matrix_size}")
    print(f"Freivalds trials per check: {trials_per_check}")
    print(f"empirical false-accept rate: {rate:.4f}")
    print(f"theoretical upper bound: {2 ** (-trials_per_check):.4f}")


if WIDGETS_AVAILABLE:
    widgets.interact(
        show_freivalds_experiment,
        matrix_size=widgets.IntSlider(value=5, min=2, max=10, step=1),
        trials_per_check=widgets.IntSlider(value=3, min=1, max=10, step=1),
        experiments=widgets.IntSlider(value=1_000, min=200, max=5_000, step=200),
    )
else:
    show_freivalds_experiment()


## Bonus: Karger's Random Contraction Algorithm

Karger's algorithm is a Monte Carlo algorithm for the global minimum cut of an undirected multigraph. It repeatedly contracts a uniformly random edge until only two supernodes remain. The crossing edges between those two supernodes are the cut returned by that run.

One run may miss the minimum cut. Repeating independent runs and keeping the smallest cut amplifies the success probability.


In [ ]:
def vertices_from_unweighted_edges(edges):
    return sorted({vertex for edge in edges for vertex in edge})


def cut_size(edges, subset):
    subset = set(subset)
    return sum((u in subset) != (v in subset) for u, v in edges)


def brute_force_min_cut(edges):
    vertices = vertices_from_unweighted_edges(edges)
    n = len(vertices)
    best_value = math.inf
    best_partitions = []

    for mask in range(1, 2 ** (n - 1)):
        subset = {vertices[i] for i in range(n) if mask & (1 << i)}
        value = cut_size(edges, subset)
        if value < best_value:
            best_value = value
            best_partitions = [subset]
        elif value == best_value:
            best_partitions.append(subset)

    return best_value, best_partitions


def karger_min_cut_once(edges, seed=SEED):
    local_rng = random.Random(seed)
    components = {vertex: {vertex} for vertex in vertices_from_unweighted_edges(edges)}
    contracted_edges = list(edges)

    while len(components) > 2:
        live_edges = [(u, v) for u, v in contracted_edges if u != v]
        u, v = local_rng.choice(live_edges)
        components[u].update(components.pop(v))

        updated_edges = []
        for a, b in contracted_edges:
            if a == v:
                a = u
            if b == v:
                b = u
            if a != b:
                updated_edges.append((a, b))
        contracted_edges = updated_edges

    return {
        "cut_size": len(contracted_edges),
        "partition": [set(part) for part in components.values()],
    }


def karger_repeated(edges, repetitions=50, seed=SEED):
    best = None
    for attempt in range(repetitions):
        result = karger_min_cut_once(edges, seed=seed + attempt)
        if best is None or result["cut_size"] < best["cut_size"]:
            best = result
    return best


sample_cut_edges = [
    ("A", "B"), ("A", "C"), ("B", "C"),
    ("D", "E"), ("D", "F"), ("E", "F"),
    ("C", "D"), ("B", "E"),
]

exact_cut_value, exact_partitions = brute_force_min_cut(sample_cut_edges)
repeated_result = karger_repeated(sample_cut_edges, repetitions=50)

print("exact min-cut value:", exact_cut_value)
print("Karger repeated result:", repeated_result)


In [ ]:
def estimate_karger_success(edges, repetitions_per_experiment=1, experiments=500, seed=SEED):
    exact_value, _ = brute_force_min_cut(edges)
    successes = 0
    for experiment in range(experiments):
        result = karger_repeated(
            edges,
            repetitions=repetitions_per_experiment,
            seed=seed + 10_000 * experiment,
        )
        successes += result["cut_size"] == exact_value
    return successes / experiments


karger_rows = []
for repetitions in (1, 2, 5, 10, 25, 50):
    karger_rows.append({
        "repetitions": repetitions,
        "empirical_success_rate": estimate_karger_success(sample_cut_edges, repetitions_per_experiment=repetitions),
    })

show_table(karger_rows)


In [ ]:
if plt is not None:
    plt.figure(figsize=(7, 4))
    plt.plot(
        [row["repetitions"] for row in karger_rows],
        [row["empirical_success_rate"] for row in karger_rows],
        "o-",
    )
    plt.xscale("log")
    plt.ylim(0, 1.05)
    plt.xlabel("independent contraction runs")
    plt.ylabel("probability of finding a true min cut")
    plt.title("Amplifying Karger's min-cut algorithm")
    plt.grid(alpha=0.25)
    plt.show()
else:
    print("matplotlib is not available, skipping plot")


## Exercises

Core exercises:

1. Use indicator random variables to prove that the expected number of hires in the basic hiring-assistant problem is $H_n$.
2. Change `hire_cost`, `fire_cost`, and `interview_cost`. Which cost dominates the expected total cost?
3. Find a threshold that reduces average cost while still hiring in at least 90% of simulated runs.

Bonus exercises:

4. Modify the secretary simulation to report the probability of selecting one of the top `k` candidates instead of only the best candidate.
5. In the adaptive hiring simulation, find a threshold that gives roughly the same final average quality as the above-mean strategy for 40 hires. Which strategy uses fewer interviews?
6. Change the Freivalds demo so the injected error changes an entire row instead of one cell. Does the empirical false-accept rate change?
7. Build a graph where one run of Karger's algorithm often fails, then estimate how many repetitions are needed to reach 95% success.


## Sources and Further Reading

- Cormen, Leiserson, Rivest, and Stein, *Introduction to Algorithms*, 4th edition, Chapter 5: [MIT Press](https://mitpress.mit.edu/9780262367509/introduction-to-algorithms/)
- MIT OpenCourseWare 6.856J, *Randomized Algorithms*, lecture notes: [OCW lecture notes](https://ocw.mit.edu/courses/6-856j-randomized-algorithms-fall-2002/pages/lecture-notes/)
- Sergei Vassilvitskii, *The Hiring Problem: Going Beyond Secretaries*: [Stanford slides](https://theory.stanford.edu/~sergei/slides/hiring-dagstuhl.pdf)
- Changyao Chen, *The Secretary Problem*: [article](https://changyaochen.github.io/secretary-problem/)
- Thomas S. Ferguson, *Who Solved the Secretary Problem?*: [PDF](https://changyaochen.github.io/assets/pdfs/who_solved_secretary_problem.pdf)
- P. R. Freeman, *The Secretary Problem and its Extensions: A Review*: [PDF](https://changyaochen.github.io/assets/pdfs/secprob2.pdf)
- Rusins Freivalds, *Probabilistic Machines Can Use Less Running Time*, IFIP Congress 1977: [DBLP record](https://dblp.org/rec/conf/ifip/Freivalds77)
- Vamsi K. Kundeti, *A Simplified Proof for the Application of Freivalds' Technique to Verify Matrix Multiplication*: [arXiv](https://arxiv.org/abs/0912.3925)
- David R. Karger, *Global Min-cuts in RNC and Other Ramifications of a Simple Min-Cut Algorithm*: [PDF](https://people.csail.mit.edu/karger/Papers/mincut.pdf)
- Joel A. Tropp, *Randomized Algorithms for Matrix Computations*: [lecture notes](https://tropp.caltech.edu/notes/Tro20-Randomized-Algorithms-LN.pdf)
